# Filter large text collections

Store scores once. Change label thresholds later without running the encoder again.
A **head** is a trained linear model that turns sentence vectors into scores.
A **mask** is one true or false value per row or label.

This recipe uses a small, deliberately label-enriched Civil Comments training sample.
It trains in scikit-learn, outside fse, then imports the finished seven-label head.
The counts below demonstrate storage and filtering. They are not quality estimates or a moderation policy.
The example is not the separately prepared publication classifier.
Use fse from master until 2.1 is released. Follow the [recipe setup](README.md#run-a-recipe).

```mermaid
flowchart LR
    T[Separate validation texts and IDs] --> H[Named linear heads]
    H --> S[All decisions and IDs on disk]
    S --> A[Read with threshold 0.5]
    S --> B[Read with threshold 0.7]
    A --> I[Selected document IDs]
    B --> J[Selected document IDs]
```

## 1. Pin the inputs and check label coverage

[Civil Comments](https://huggingface.co/datasets/google/civil_comments/tree/f2970eb3a55777454c94069077cc8d9b5866312d) is released under CC0.
Its seven targets are fractions of annotators assigning each attribute, as described by [TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/civil_comments).
Training marks a target positive at `>= 0.5`, including equality.
The classifier's later decision uses **strictly greater than** its threshold.
These model outputs describe binarized targets. They are not calibrated estimates of annotator agreement.
The data contains offensive language. We display only counts and IDs, never comment text.

The fixed sample uses the first 2,048 training rows plus the first 16 positive and negative rows per label from the first training shard.
The union stays in source order. This selection ensures both classes exist for every label and deliberately changes prevalence.
Demonstration texts are the first 256 validation rows, never training rows.
No test split, tuning search or quality comparison is used.


In [ ]:
from _helpers import Recipe, POTION_REVISION
import fse
import numpy as np
import pyarrow.parquet as pq

recipe = Recipe()
work = recipe.work
REVISION = "f2970eb3a55777454c94069077cc8d9b5866312d"
LABELS = ["toxicity", "severe_toxicity", "obscene", "threat", "insult", "identity_attack", "sexual_explicit"]
SEED = 17
print("Civil Comments revision:", REVISION)
print("POTION-base-8M revision:", POTION_REVISION)

train_file = recipe.source_file("google/civil_comments", REVISION, "data/train-00000-of-00002.parquet", repo_type="dataset")
validation_file = recipe.source_file("google/civil_comments", REVISION, "data/validation-00000-of-00001.parquet", repo_type="dataset")
target_table = pq.read_table(train_file, columns=LABELS)
targets = np.column_stack([target_table[name].to_numpy() >= 0.5 for name in LABELS])
chosen = set(range(2048))
for j, name in enumerate(LABELS):
    positive = np.flatnonzero(targets[:, j])
    negative = np.flatnonzero(~targets[:, j])
    assert len(positive) >= 16 and len(negative) >= 16, name
    chosen.update(positive[:16].tolist())
    chosen.update(negative[:16].tolist())
train_positions = np.array(sorted(chosen), dtype=np.int64)
y_train = targets[train_positions].astype(np.int64)
del targets, target_table
train_texts = pq.read_table(train_file, columns=["text"]).take(train_positions)["text"].to_pylist()
validation = next(pq.ParquetFile(validation_file).iter_batches(batch_size=256))
texts = validation.column("text").to_pylist()
y_demo = np.column_stack([validation.column(name).to_numpy() >= 0.5 for name in LABELS])
# Application IDs name validation positions here, not identities from the dataset.
doc_ids = np.arange(len(texts), dtype=np.int64) + 1000000
assert len(set(train_texts).intersection(texts)) == 0
assert np.all(y_train.sum(axis=0) > 0) and np.all(y_train.sum(axis=0) < len(y_train))
print("Training rows:", len(y_train), "Separate demonstration rows:", len(texts))
print("Training positives:", dict(zip(LABELS, y_train.sum(axis=0).tolist())))
print("Training negatives:", dict(zip(LABELS, (len(y_train) - y_train.sum(axis=0)).tolist())))
print("Demonstration positives:", dict(zip(LABELS, y_demo.sum(axis=0).tolist())))


## 2. Train outside fse and import the head

POTION-base-8M supplies fixed token vectors under MIT. Its revision is pinned by the [shared helper](_helpers.py).
Load raw means with `normalize=False`, so the linear head can fold into token scores.
A standard scaler centers and scales the features. One logistic regression per label learns an independent binary decision.
fse imports that scaler into the linear weights. It trains nothing.
Settings are fixed: seed 17, C=1, one job, no class weights, at most 1,000 iterations.
A convergence warning stops this demonstration instead of accepting an unfinished fit.
Explicit `labels=` preserves the names and their column order.


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

encoder = fse.Encoder.from_pretrained(recipe.model_folder(), normalize=False)
training = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(LogisticRegression(C=1, solver="lbfgs", max_iter=1000, random_state=SEED), n_jobs=1),
)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    training.fit(encoder.encode(train_texts, workers=1), y_train)
toxic = fse.Classifier.from_sklearn(encoder, training, labels=LABELS)
assert list(toxic.classes_) == LABELS
# A second task reuses the same encoder: flag any of the seven training targets.
any_model = LogisticRegression(C=1, max_iter=1000, random_state=SEED)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    any_model.fit(encoder.encode(train_texts, workers=1), y_train.any(axis=1))
any_flag = fse.Classifier.from_sklearn(encoder, any_model, labels=["unflagged", "flagged"])
heads = fse.Heads(toxic=toxic, any_flag=any_flag)
print("Class order:", list(toxic.classes_))
print("Score-only paths:", dict(heads.paths()))


## 3. Save every score with its document ID

A **decision** is a logit, the number before the probability rule.
Store decisions so fse can later derive probabilities and masks with its own numerical rules.
Saving only masks would lose the information needed for other thresholds.
The batches below carry `(texts, doc_ids)`, so set `doc_ids=True` on both the sink and the call.
Only the scores are stored here, and raw-mean heads take the folded path.


In [ ]:
def batches():
    for start in range(0, len(texts), 64):
        yield texts[start:start + 64], doc_ids[start:start + 64]

score_path = work / "all-scores"
with fse.ScoreSink(score_path, heads=heads, doc_ids=True, shard_rows=100) as sink:
    for batch in heads.predict_batches(batches(), doc_ids=True, score_sink=sink, workers=1):
        pass

# Reopen after the writer closes, checking model identities and stored paths.
scores = fse.ScoreStore.open(score_path, heads=heads)
assert len(scores) == len(texts) and scores.has_doc_ids
assert list(scores.heads["toxic"].labels) == LABELS
assert scores.heads["toxic"].path == "folded"
np.testing.assert_array_equal(scores.get_doc_ids(), doc_ids)
folded = heads.predict(texts, output="decision", workers=1)
for name in heads:
    np.testing.assert_array_equal(scores.get_scores(name, "decision").view(np.uint32), folded[name].view(np.uint32))
scores.verify()
print("Reopened and verified score rows:", len(scores))


## 4. Try stricter thresholds without new inference

`ScoreStore.predict_batches` reads the stored decisions and derives masks without an encoder.
A scalar threshold replaces every threshold for that head. A named mapping changes selected labels only.
The default is 0.5 per label. Equality produces false.
We check each recovered mask against the same fse head with `with_thresholds`.
No NumPy sigmoid substitutes for fse's probability rule.


In [ ]:
def masks(threshold):
    blocks = list(scores.predict_batches(thresholds={"toxic": threshold}, block_rows=73))
    np.testing.assert_array_equal(np.concatenate([b.doc_ids for b in blocks]), doc_ids)
    return np.concatenate([b.outputs["toxic"] for b in blocks])

soft = masks(0.5)
strict = masks(0.7)
for threshold, actual in ((0.5, soft), (0.7, strict)):
    expected = toxic.with_thresholds(threshold).predict(texts, workers=1)
    np.testing.assert_array_equal(actual, expected)
assert not np.any(strict & ~soft)

# An actual float32 probability is used as its own cutoff: equality is false.
probability = toxic.predict_proba(texts[:1], workers=1)[0, 0]
cutoff = {"toxicity": float(probability)}
np.testing.assert_array_equal(masks(cutoff), toxic.with_thresholds(cutoff).predict(texts, workers=1))
assert not masks(cutoff)[0, 0]

print("Demonstration flags above 0.5:", dict(zip(LABELS, soft.sum(axis=0).tolist())))
print("Demonstration flags above 0.7:", dict(zip(LABELS, strict.sum(axis=0).tolist())))
print("Example IDs flagged above 0.5:", doc_ids[soft.any(axis=1)][:10].tolist())


## 5. A discarded row cannot come back from a subset

Here, select rows flagged by at least one label. A stricter threshold selects a subset of the softer one.
If you retained only those strict rows, a later softer threshold could only inspect that subset.
It cannot recover discarded rows. The full score store still contains them all.
In an application, join these IDs to the source records to choose which texts to retain or review.


In [ ]:
strict_rows = strict.any(axis=1)
soft_rows = soft.any(axis=1)
strict_ids = doc_ids[strict_rows]
soft_ids_from_subset = strict_ids[soft[strict_rows].any(axis=1)]
soft_ids_from_all = doc_ids[soft_rows]
np.testing.assert_array_equal(soft_ids_from_subset, strict_ids)
assert set(soft_ids_from_subset).issubset(soft_ids_from_all)
missing = np.setdiff1d(soft_ids_from_all, soft_ids_from_subset)
print("Rows selected from all scores at 0.7:", len(strict_ids))
print("Rows selected from all scores at 0.5:", len(soft_ids_from_all))
print("Additional IDs unavailable from the strict subset:", len(missing))


## 6. Write vectors and scores in one pass

Use a `ShardSink` for vectors and a `ScoreSink` for scores, with different directories.
Both receive the same document IDs. `durable_rows` confirms the smaller durable prefix of the run.
Each store is crash-safe separately. There is no atomic commit across both folders.
For interruptions and replay, see [document IDs](../docs/document-ids.md) and the [streaming recipe](09-fse-behind-kafka.ipynb).

`return_vectors=True` requires full sentence vectors, so the raw-mean heads take the **unfolded** path.
Compare stored decisions only with inference on that same path.
Folding changes rounding order, so folded and unfolded decisions do not promise bit identity.
The [numerical contract](../docs/architecture/numerical-contract.md) explains their relationship.


In [ ]:
vector_path = work / "vectors"
combined_score_path = work / "scores-with-vectors"
with fse.ShardSink(vector_path, encoder, doc_ids=True, shard_rows=100) as vector_sink, fse.ScoreSink(
    combined_score_path, heads=heads, doc_ids=True, shard_rows=100
) as score_sink:
    confirmed = 0
    for batch in heads.predict_batches(
        batches(), doc_ids=True, return_vectors=True,
        vector_sink=vector_sink, score_sink=score_sink, workers=1,
    ):
        assert batch.durable_rows is not None and batch.durable_rows >= confirmed
        confirmed = batch.durable_rows

reference_batches = list(heads.predict_batches([texts], output="decision", return_vectors=True, workers=1))
unfolded = {name: np.concatenate([b.outputs[name] for b in reference_batches]) for name in heads}
reference_vectors = np.concatenate([b.vectors for b in reference_batches])
with fse.VectorStore.open(vector_path) as vectors, fse.ScoreStore.open(combined_score_path, heads=heads) as combined:
    assert len(vectors) == len(combined) == len(doc_ids)
    assert combined.heads["toxic"].path == "unfolded"
    for store in (vectors, combined):
        np.testing.assert_array_equal(store.get_doc_ids(), doc_ids)
        store.verify()
    np.testing.assert_array_equal(vectors.get_vectors().view(np.uint32), reference_vectors.view(np.uint32))
    for name in heads:
        np.testing.assert_array_equal(combined.get_scores(name, "decision").view(np.uint32), unfolded[name].view(np.uint32))
print("Verified vectors, scores and matching IDs:", len(doc_ids))


## 7. Save the finished head separately

A group has no bundle format. Save each head on its own.
This complete classifier retains its encoder and named labels.
Reloading must preserve the folded decision bits and class order.
Use permanent paths for your own models. This recipe's files are temporary and nothing is uploaded.


In [ ]:
model_path = work / "demo-classifier.safetensors"
toxic.save(model_path)
loaded = fse.Classifier.load(model_path)
assert list(loaded.classes_) == LABELS
np.testing.assert_array_equal(
    loaded.decision_function(texts, workers=1).view(np.uint32), folded["toxic"].view(np.uint32)
)
print("Reloaded labels in the same order:", list(loaded.classes_))


## Limits and next steps

These enriched, small training rows do not establish moderation quality.
Targets reflect annotator judgments on English news comments from 2015 to 2017.
Rare labels and identity bias need separate evaluation. Do not automate moderation based on these demonstration counts.
Choose any application thresholds on separate validation data and evaluate your final policy before deployment.
A stored mask is a model decision, not a fact about a person or text.

- [Several heads and classifier training methods](02-linear-heads.ipynb)
- [Folding and model size](08-folded-classifier.ipynb)
- [Score-store format and guarantees](../docs/architecture/scores.md)
- [Recipe index](README.md)

## Clean up

Close mappings before removing the temporary workspace.


In [ ]:
scores.close()
recipe.close()
